# YouTube Playlist — Top 10 by Views
Run each cell top to bottom. Paste your URLs into the input box that appears in Step 2.

In [ ]:
!pip install requests ipywidgets --quiet

## Step 1 — API Key

In [ ]:
API_KEY = "YOUR_API_KEY_HERE"  # https://console.cloud.google.com/ → YouTube Data API v3
TOP_N = 10

## Step 2 — Load helpers
Run this cell as-is (no edits needed).

In [ ]:
import re, requests

YT_API = "https://www.googleapis.com/youtube/v3"

def is_channel_url(url):
    return bool(re.search(r"youtube\.com/(@|channel/|user/)", url))

def get_channel_info(url):
    handle  = re.search(r"youtube\.com/@([A-Za-z0-9_.-]+)", url)
    channel = re.search(r"youtube\.com/channel/(UC[A-Za-z0-9_-]+)", url)
    user    = re.search(r"youtube\.com/user/([A-Za-z0-9_.-]+)", url)
    if handle:
        params = {"part": "snippet,contentDetails", "forHandle": handle.group(1), "key": API_KEY}
    elif channel:
        params = {"part": "snippet,contentDetails", "id": channel.group(1), "key": API_KEY}
    elif user:
        params = {"part": "snippet,contentDetails", "forUsername": user.group(1), "key": API_KEY}
    else:
        raise ValueError(f"Could not parse channel URL: {url}")
    resp = requests.get(f"{YT_API}/channels", params=params, timeout=15)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    if not items:
        raise ValueError(f"No channel found for: {url}")
    return items[0]["snippet"]["title"], items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

def get_playlist_name(playlist_id):
    resp = requests.get(f"{YT_API}/playlists", params={"part": "snippet", "id": playlist_id, "key": API_KEY}, timeout=15)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    return items[0]["snippet"]["title"] if items else playlist_id

def get_all_video_ids(playlist_id):
    video_ids, page_token = [], None
    while True:
        params = {"part": "contentDetails", "playlistId": playlist_id, "maxResults": 50, "key": API_KEY}
        if page_token:
            params["pageToken"] = page_token
        data = requests.get(f"{YT_API}/playlistItems", params=params, timeout=15).json()
        video_ids += [i["contentDetails"]["videoId"] for i in data.get("items", []) if i["contentDetails"].get("videoId")]
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    return video_ids

def get_view_counts(video_ids):
    views = []
    for i in range(0, len(video_ids), 50):
        params = {"part": "statistics", "id": ",".join(video_ids[i:i+50]), "key": API_KEY}
        for item in requests.get(f"{YT_API}/videos", params=params, timeout=15).json().get("items", []):
            views.append(int(item.get("statistics", {}).get("viewCount", 0)))
    return views

def process_url(url):
    if is_channel_url(url):
        name, playlist_id = get_channel_info(url)
        print(f"  Channel : {name}")
    else:
        m = re.search(r"[?&]list=([A-Za-z0-9_-]+)", url)
        if not m:
            raise ValueError(f"Could not find playlist ID in: {url}")
        playlist_id = m.group(1)
        name = get_playlist_name(playlist_id)
        print(f"  Playlist: {name}")
    video_ids = get_all_video_ids(playlist_id)
    print(f"  {len(video_ids)} videos — fetching view counts...")
    views = sorted(get_view_counts(video_ids), reverse=True)
    return name, views[:TOP_N]

print("Helpers loaded.")

## Step 3 — Paste URLs
Run this cell to show the input box, paste your URLs in (one per line), then click **Run**.

In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display, clear_output

url_box = widgets.Textarea(
    placeholder="Paste URLs here, one per line...",
    layout=widgets.Layout(width="100%", height="160px"),
)
run_btn = widgets.Button(description="Run", button_style="primary")
out = widgets.Output()

df = pd.DataFrame()

def on_run(_):
    global df
    urls = [u.strip() for u in url_box.value.strip().splitlines() if u.strip()]
    with out:
        clear_output()
        if not urls:
            print("No URLs entered.")
            return
        rows = []
        for url in urls:
            print(f"Processing: {url}")
            try:
                name, top_views = process_url(url)
                rows += [{"playlist_name": name, "views": v} for v in top_views]
            except Exception as e:
                print(f"  ERROR: {e}")
        df = pd.DataFrame(rows)
        display(df)

run_btn.on_click(on_run)
display(url_box, run_btn, out)

## Step 4 — Download CSV
Run after Step 3 has finished.

In [ ]:
from google.colab import files

df.to_csv("top10_views.csv", index=False)
files.download("top10_views.csv")